# 01. Preparación y revisión de datos

Este notebook es el punto de partida. Está pensado para revisar que las bases principales existen, tienen el formato esperado y contienen las columnas necesarias para reproducir el modelo PAMPA de la tesis.

## Qué debe cambiar el usuario

Si quieres usar otra base de datos, cambia las rutas en la celda **Configuración editable**. Para reproducir la tesis, no cambies nada.

El modelo final necesita tres archivos:

- entrenamiento: `data/raw/training_11.csv`
- prueba interna: `data/raw/test_11.csv`
- validación externa: `data/raw/external_11.csv`

Cada archivo debe tener los 11 descriptores seleccionados y la columna `Actividad` con etiquetas `Act1` o `Act-1`.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
%cd $ROOT

## Configuración editable

Cambia estas rutas solo si vas a revisar otro conjunto de datos. Si el objetivo es reproducir la tesis, deja las rutas por defecto.

In [ ]:
TRAIN_FILE = Path('data/raw/training_11.csv')
TEST_FILE = Path('data/raw/test_11.csv')
EXTERNAL_FILE = Path('data/raw/external_11.csv')

REQUIRED_DESCRIPTORS = [
    'LOGPcons', 'MACCSFP125', 'PCR', 'Psi_e_A', 'P_VSA_ppp_D',
    'Mp', 'SpMin1_Bh(p)', 'SHED_AL', 'SM12_AEA(ri)',
    'P_VSA_s_3', 'MATS5m'
]
TARGET_COLUMN = 'Actividad'

## Carga de bases

Esta celda carga las tres bases y muestra cuántas filas/columnas tiene cada una.

In [ ]:
datasets = {
    'training': pd.read_csv(TRAIN_FILE),
    'test_internal': pd.read_csv(TEST_FILE),
    'external': pd.read_csv(EXTERNAL_FILE),
}

pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'columns': df.shape[1]}
    for name, df in datasets.items()
])

## Validación de columnas

Aquí se revisa si falta algún descriptor. Si aparece una lista vacía `[]`, está correcto.

In [ ]:
checks = []
for name, df in datasets.items():
    missing = [col for col in REQUIRED_DESCRIPTORS + [TARGET_COLUMN] if col not in df.columns]
    checks.append({'dataset': name, 'missing_columns': missing, 'valid': len(missing) == 0})
pd.DataFrame(checks)

## Distribución de clases

`Act1` representa moléculas permeables. `Act-1` representa moléculas no permeables. La base externa está desbalanceada, lo cual explica por qué el modelo prioriza sensibilidad.

In [ ]:
class_summary = []
for name, df in datasets.items():
    counts = df[TARGET_COLUMN].value_counts().to_dict()
    class_summary.append({
        'dataset': name,
        'Act1_permeable': counts.get('Act1', 0),
        'Act-1_non_permeable': counts.get('Act-1', 0),
    })
pd.DataFrame(class_summary)

## Vista rápida de la base

Esta tabla permite comprobar visualmente que los valores son numéricos y que la columna `Actividad` está al final.

In [ ]:
datasets['training'].head()

## Salida esperada

Si todo está correcto:

- las tres bases cargan sin error;
- `missing_columns` aparece vacío;
- existen etiquetas `Act1` y `Act-1`;
- puedes continuar al notebook `02_Entrenamiento_RandomForest.ipynb`.